In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in {'Baseline_Training', 'CTC_models', 'PhoneticFeatures_Training'}:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from project_paths import (
    BASELINE_MODELS_DIR,
    CTC_MODELS_DIR,
    DEFAULT_BASELINE_MODEL,
    DEFAULT_CTC_ATTENTION_MODEL,
    DEFAULT_CTC_MODEL,
    DEFAULT_PHONETIC_MODEL_2CLASS,
    DEFAULT_PHONETIC_MODEL_3CLASS,
    EXAMPLE_EMBEDDINGS,
    EXAMPLE_PHONEMES,
    EXAMPLE_SEG_B2,
    EXAMPLE_SEG_B4,
    EXAMPLE_WAV,
    PHONEME_CHOICES_SUMMARY,
    PHONETIC_MODELS_DIR,
    TABLES_DIR,
)


In [ ]:
import panphon
import pandas as pd
import matplotlib.pyplot as plt
import csv


### Example of conversion word to features

In [ ]:
ft = panphon.FeatureTable()
word = "sɑlti"
fts = ft.word_fts(word)
seg = fts[0]

print("Признаки (ключи) первого сегмента:")
print(list(seg))

## Choice of symbols

In [ ]:
# пути к файлам
ipa_all_path = r'C:\Users\ext-ananeva\venv3.7\Lib\site-packages\panphon\data\ipa_all.csv'   # поменяй под свой путь
ipa_rus_path = r"C:\Users\ext-ananeva\venv3.7\Lib\site-packages\panphon\data\ipa_rus.csv"
ipa_reduced = r"C:\Users\ext-ananeva\venv3.7\Lib\site-packages\panphon\data\ipa_reduced.csv"
ipa_selected = r"C:\Users\ext-ananeva\venv3.7\Lib\site-packages\panphon\data\ipa_selected.csv"


In [ ]:
df_all = pd.read_csv(ipa_all_path, index_col=0)

mapping = {"-": -1, "+": 1, "0": 0}
df_all = df_all.replace(mapping)

### Get description of a symbol

In [ ]:
idx = "l" 

row_dict = df_all.loc[idx].to_dict()
row_dict

### Get symbol with description

In [ ]:
row_dict['hi']=-1
find_features = {'syl': -1,
 'son': -1,
 'cons': 1,
 'cont': 1,
 'delrel': -1,
 'lat': -1,
 'nas': -1,
 'strid': 1,
 'voi': 1,
 'sg': -1,
 'cg': -1,
 'ant': -1,
 'cor': 1,
 'distr': 1,
 'lab': -1,
 'hi': 1,
 'lo': -1,
 'back': -1,
 'round': -1,
 'velaric': -1,
 'tense': 0,
 'long': -1,
 'hitone': 0,
 'hireg': 0}
# find_features = row_dict
find_features = {'syl': -1, 'son': 1, 'cons': 1, 'cont': 1, 'delrel': -1, 'lat': -1, 'nas': -1, 'strid': -1, 'voi': 1, 'sg': -1, 'cg': -1, 'ant': 1, 'cor': 1, 'distr': -1, 'lab': -1, 'hi': 1, 'lo': -1, 'back': -1, 'round': -1, 'velaric': -1, 'tense': -1, 'long': -1, 'hitone': 0, 'hireg': 0}

mask = (df_all[list(find_features)] == pd.Series(find_features)).all(axis=1)
result = df_all[mask]
result

In [ ]:
symbols = []
target_list = []
for symb in df_all.index.to_list():
    if not any(s in symb for s in [ 'ǃ', '˩', '̘', 'ɮ', 'ˡ', 'ʘ', '̆', 'ᶣ', '̃', '̞', '̯', 'ɔ', 'ˤ', 'ː', 'ǁ', 'ǂ', 'ɬ', 'ˠ', '̪', '˦', '̥', 'ʰ', '̠', '̻', '͡', '̈', '̟', 'ˀ', '̧', '̩', '̝', '˥', '˞', '̤', 'ʷ',  '̺', '̙', '̴', '̰', 'ⁿ', '˨', 'ʼ', '̼', 'ʍ', 'ɹ', 'ǀ', '˧']):
        target_list.append(symb)


In [ ]:
len(target_list)

In [ ]:
psymbols = []

for symb in df_all.index.to_list():
    for s in symb:
        symbols.append(s)

In [ ]:
print(set(symbols))

## Get russian sounds

In [ ]:

with open(ipa_all_path, "r", encoding="utf-8") as fin, \
     open(ipa_reduced, "w", encoding="utf-8", newline="") as fout:

    reader = csv.reader(fin)
    writer = csv.writer(fout)

    # копируем заголовок (первая строка)
    header = next(reader)
    writer.writerow(header)
    print(header)
    

    # фильтруем строки по символу
    for row in reader:
        if row and row[0] in target_list:
            if row[0] in ['k', 'ɡ', 'x']:
                change_feature_index = header.index('hi')
                
                row[change_feature_index] = '-'

            writer.writerow(row)

print("Файл ipa_reduced.csv создан, содержит только нужные символы.")

In [ ]:
symbols_rus = [
    # Гласные
    "i", "ɨ", "u", "e", "o", "ɛ", "a", "ə", "ʌ",

    # Согласные
    "p", "b",
    "t", "d",
    "k", "ɡ",
    "f", "v",
    "s", "z",
    "ʂ", "ʐ",
    "t͡s", "d͡z",
    "t͡ɕ", "d͡ʑ",
    "x", "ɣ",
    "m", "n", "l", "r",
    "j","ɺ", 'ɾ',

    # Палатализованные согласные
    "pʲ", "bʲ",
    "tʲ", "dʲ",
    "kʲ", "ɡʲ",
    "fʲ", "vʲ",
    "sʲ", "zʲ",
    "ɕ", "ʑ",
    "xʲ",
    "mʲ", "nʲ", "lʲ", "rʲ", "ɺʲ","ɾʲ",]


with open(ipa_all_path, "r", encoding="utf-8") as fin, \
     open(ipa_rus_path, "w", encoding="utf-8", newline="") as fout:

    reader = csv.reader(fin)
    writer = csv.writer(fout)

    # копируем заголовок (первая строка)
    header = next(reader)
    writer.writerow(header)
    print(header)
    

    # фильтруем строки по символу
    for row in reader:
        if row and row[0] in symbols_rus:
            if row[0] in ['k', 'ɡ', 'x']:
                change_feature_index = header.index('hi')
                
                row[change_feature_index] = '-'

            writer.writerow(row)

print("Файл ipa_rus.csv создан, содержит только нужные символы.")

In [ ]:
import pandas as pd

df = pd.read_csv(ipa_all_path)
#df = df[:1000]
feature_cols = [col for col in df.columns if col != "ipa"]

In [ ]:
from collections import Counter
import unicodedata

# -------------------------
# ПРИОРИТЕТЫ
# -------------------------

DIACRITIC_PRIORITY = {"ʷ": 2, "ʲ": 2, "ˀ": -1}

BASE_PRIORITY = {
    "ə": 5,
    "ɐ": 4,
    "e": 3,
    "i": 2,
    "ɨ": 2,
    "N": -1,
    "r": 3,
    "ɾ": 2,
    "o": 3,
    "ɒ": 2
}

IGNORED_DIACRITICS = {"ʷ", "ʲ", "ˀ", "̰", "̠"}

# -------------------------
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# -------------------------

def split_base_and_diacritics(s: str):
    base = []
    diacritics = []

    for ch in s:
        # combining marks
        if unicodedata.combining(ch) != 0:
            diacritics.append(ch)
        elif ch in IGNORED_DIACRITICS:
            diacritics.append(ch)
        else:
            base.append(ch)

    return "".join(base), diacritics


def score_candidate(s: str):
    base, diacritics = split_base_and_diacritics(s)

    score = 0

    # ---- база ----
    for b in base:
        score += BASE_PRIORITY.get(b, 0)

    # ---- диакритики ----
    for d in diacritics:
        score += DIACRITIC_PRIORITY.get(d, 0)

    if any(d in ['̃', '̤', "ˀ",'ˠ','ᶣ'] for d in diacritics):
        score -= 5
        
    # ---- бонус за ʷ / ʲ ----
    if any(d in ["ʷ", "ʲ", '̺', '̪', 'ʰ', 'ʰ'] for d in diacritics):
        score += 3


    scores = {
        "ə": 2, "ɐ": 2, "e": -1,
        "r": 2, "ɾ": -1,
        "o": 2, "ɒ": -1,
        "k": 2, "q": -1,
        "ɡ": 2, "ɢ": -1,
        "ɠ": 2, "ʛ": -1,
        "ʃ": 2, "ɕ": -1,
        "N": -2, "ɴ": -2,
        "ʑ": 2, "ʒ": -1,
        "ɺ": 2, "ɫ": -1,
        "æ": 2, "ɛ": -1,
        "œ": 2, "ɶ": -1,
        "ø": 2, "ʏ": -1,
        'i': 5
       
       
    }
    
    score += sum(scores.get(char, 0) for char in base)

    return score


def colon_order_score(s: str):
    """
    Приоритет:
    colon должен быть ДО диакритиков
    """
    if "ː" not in s:
        return 0

    colon_idx = s.index("ː")

    # позиции диакритиков (упрощённо: всё кроме base и 🙂
    _, diacritics = split_base_and_diacritics(s)

    # если colon стоит раньше всех диакритиков → хорошо
    diacritic_positions = [s.index(d) for d in diacritics if d in s]

    if not diacritic_positions:
        return 1

    return 1 if colon_idx < min(diacritic_positions) else -1


# -------------------------
# ОСНОВНАЯ ФУНКЦИЯ
# -------------------------

def choose_phoneme(candidates):
    if not candidates:
        return None

    # 1. минимальная длина
    min_len = min(len(c) for c in candidates)
    candidates = [c for c in candidates if len(c) == min_len]

    if len(candidates) == 1:
        return candidates[0]

    # 2. скоринг
    scored = [(c, score_candidate(c)) for c in candidates]
    scored.sort(key=lambda x: x[1], reverse=True)

    best_score = scored[0][1]
    top = [c for c, s in scored if s == best_score]

    if len(top) == 1:
        return top[0]
    
    # 3. правило порядка диакритиков (твоя логика)

    def find_candidate(candidates, diacritics):
        if not candidates:
            return None
    
        # 1. проверяем одинаковый набор символов
        base_set = set(candidates[0])
    
        for c in candidates[1:]:
            if set(c) != base_set:
                return None
    
        # 2. приоритет диакритиков
        for d in diacritics:
            for c in candidates:
                if c.endswith(d):
                    return [c]    
        return None
    
    
    # применяем правило
    res = find_candidate(top, ['ː', "ː",'ʰ', "ˀ", "ʲ", "ʷ", 'ʰ', 'ɮ'])
    
    if res is not None and len(res) == 1:
        return res[0]
    if res is not None:
        top = res
        
        

    options = " | ".join(f"{i}:{ph}" for i, ph in enumerate(top))
    
    while True:
        choice = input(f"\nНеоднозначно ({options}): ")
        if choice.isdigit() and int(choice) < len(candidates):
            return candidates[int(choice)]
        print("Ошибка, попробуй снова")

In [ ]:
# =========================
# 2. Выбор фонем
# =========================
selected = {}
summary_rows = []

for _, group in df.groupby(feature_cols):
    phonemes = group["ipa"].tolist()
    key = tuple(group.iloc[0][feature_cols])

    chosen = choose_phoneme(phonemes)
    selected[key] = chosen

    summary_rows.append({
        **dict(zip(feature_cols, key)),
        "all_variants": ", ".join(phonemes),
        "selected": chosen
    })


In [ ]:
# =========================
# 3. Применяем выбор ко всему датасету
# =========================
def get_selected(row):
    key = tuple(row[feature_cols])
    return selected.get(key, row["ipa"])

df["ipa_selected"] = df.apply(get_selected, axis=1)


In [ ]:
# 4. Финальная таблица (без дублей)
# =========================
result = df.drop_duplicates(subset=feature_cols)
result = result.rename(columns={"ipa_selected": "iiiii"})
result = result[["ipa"] + feature_cols]

result.to_csv(ipa_selected, index=False)



In [ ]:
df = pd.read_csv(ipa_selected, index_col=0)

In [ ]:
# =========================
# 5. Таблица "признаки | варианты | выбранный"
# =========================
summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(PHONEME_CHOICES_SUMMARY, index=False)



In [ ]:
selected_df = pd.read_csv(PHONEME_CHOICES_SUMMARY)
len(selected_df)

In [ ]:
selected_symbols = list(selected_df['selected'])


with open(ipa_all_path, "r", encoding="utf-8") as fin, \
     open(ipa_selected, "w", encoding="utf-8", newline="") as fout:

    reader = csv.reader(fin)
    writer = csv.writer(fout)

    # копируем заголовок (первая строка)
    header = next(reader)
    writer.writerow(header)
    print(header)
    

    # фильтруем строки по символу
    for row in reader:
        if row and row[0] in selected_symbols:

            writer.writerow(row)

print("Файл ipa_selected.csv создан, содержит только нужные символы.")

In [ ]:
# =========================
# 6. Проверка
# =========================
print("\nГотово!")
print(f"Исходных строк: {len(df)}")
print(f"Уникальных комбинаций: {len(result)}")

### Visualize

In [ ]:
df = pd.read_csv(ipa_rus_path, index_col=0)
df = df.replace(mapping)

df = df.drop('hitone', axis=1)
df = df.drop('hireg', axis=1)

plt.figure(figsize=(34, 25))
plt.rcParams.update({'font.size': 35})

plt.imshow(df.values, cmap="RdBu", aspect="auto", vmin=-1, vmax=1)

# plt.colorbar(label="Значение признака")

plt.xticks(range(len(df.columns)), df.columns, rotation=45)

plt.yticks(range(len(df.index)), df.index)

plt.title("Фонетические признаки, соответвтвующие символам IPA")
plt.tight_layout()
plt.show()


In [ ]:
selected_symbols = [ 'a', 'b', 'p' ]  # пример

# Оставляем только выбранные строки (символы)
df_filtered = df.loc[df.index.isin(selected_symbols)]

plt.figure(figsize=(34, 5))
plt.rcParams.update({'font.size': 30})

plt.imshow(df_filtered.values, cmap="RdBu", aspect="auto", vmin=-1, vmax=1)

plt.colorbar(label="Значение признака")

plt.xticks(range(len(df_filtered.columns)), df_filtered.columns, rotation=45)

plt.yticks(range(len(df_filtered.index)), df_filtered.index)

plt.title("Тепловая карта признаков IPA (только выбранные символы)")
plt.tight_layout()
plt.show()


In [ ]:
for i, row in df.iterrows():

    find_festures = row.to_list()

    for j, row2 in df.iterrows():
        feats = row2.to_list()
        if find_festures == feats and j!=i:
            print(i)
            
        

In [ ]:
df

# Расшифровка
`['syl', 'son', 'cons', 'cont', 'delrel', 'lat', 'nas', 'strid', 'voi', 'sg', 'cg', 'ant', 'cor', 'distr', 'lab', 'hi', 'lo', 'back', 'round', 'velaric', 'tense', 'long']`

- **`syl`** — **syllabic**  
  `+` = слоговой (обычно гласные и слоговые сонорные, например `n̩`, `l̩`) (**В русском только гласные**); 
  `-` = не‑слоговой (обычные согласные).

- **`son`** — **sonorant**  
  `+` = сонорный (гласные, носовые, латерали, вибранты, аппроксиманты);  
  `-` = шумный (стопы, фрикативы, аффрикаты).

- **`cons`** — **consonantal**  
  `+` = согласный (обычные согласные);  
  `-` = не‑согласный (гласные, иногда глиды).

- **`cont`** — **continuant**  
  `+` = непрерывный (можно тянуть: фрикативы, аппроксиманты, гласные);  
  `-` = прерывистый (стопы, аффрикаты, носовые).

- **`delrel`** — **delayed release**  
  `+` = есть задержка/смычка с последующим постепенным открытием (аффрикаты);  
  `-` = нет задержки (обычные стопы, фрикативы).

- **`lat`** — **lateral**  
  `+` = латеральный (воздух идёт по бокам языка, например `l`, `ɬ`);  
  `-` = центральный.

- **`nas`** — **nasal**  
  `+` = назальный (носовой, например `m`, `n`, `ŋ`);  
  `-` = оральный.

- **`strid`** — **strident**  
  `+` = шумный, «шипящий» (сильные фрикативы/аффрикаты, например `s`, `ʃ`, `z`, `ʒ`);  
  `-` = не‑шипящий.

- **`voi`** — **voice**  
  `+` = озвончённый (голосовые связки вибрируют);  
  `-` = глухой.

- **`sg`** — **spread glottis**  
  `+` = голосовая щель разведена, есть аспирация (аспираты, аспирированные согласные);  
  `-` = нет аспирации.

- **`cg`** — **constricted glottis**  
  `+` = голосовая щель сужена (глоттализация, эйджитивы, имплозивы);  
  `-` = нет сужения.

- **`ant`** — **anterior**  
  `+` = передний (артикуляция в передней части рта: губы, зубы, альвеолы);  
  `-` = не‑передний (задние, например велярные, увулярные).

- **`cor`** — **coronal**  
  `+` = корональный (язык участвует активно: зубные, альвеолярные, палато‑альвеолярные);  
  `-` = не‑корональный (лабиальные, велярные и т.п.).

- **`distr`** — **distributed**  
  `+` = широкий контакт языка (пальмальные/широкие корональные, например `ʃ`, `ʒ`);  
  `-` = узкий контакт (узкие корональные, например `s`, `z`).

- **`lab`** — **labial**  
  `+` = лабиальный (артикуляция с участием губ);  
  `-` = не‑лабиальный.

- **`hi`** — **high**  
  `+` = высокий (язык высоко: `i`, `u`, `ɪ`, `ʊ`) у согласных - мягкость;  
  `-` = не‑высокий.

- **`lo`** — **low**  
  `+` = низкий (язык низко: `a`, `ɑ`);  
  `-` = не‑низкий.

- **`back`** — **back**  
  `+` = задний (корень языка смещён назад: `u`, `o`, `ɑ`);  
  `-` = передний/центральный.

- **`round`** — **round**  
  `+` = округлённый (губы округлены: `u`, `o`, `y`, `ø`);  
  `-` = не‑округлённый.

- **`velaric`** — **velaric**  
  `+` = клик (вельярный/устьевой клик, например `ʘ`, `ǀ`, `ǃ`, `ǂ`, `ǁ`);  
  `-` = не‑клик.

- **`tense`** — **tense**  
  `+` = напряжённый (язык напряжён, «чистый» гласный, например `i`, `u`);  
  `-` = расслабленный (например `ɪ`, `ʊ`).

- **`long`** — **long**  
  `+` = долгий (удвоенный по времени, например `aː`, `lː`);  
  `-` = краткий.


## Требует изменения 
- у g, x нет мягкой пары
- можно убрать velaric, long, sg, cg, hitone, hireg
- tense как альтернатива ударности (?)
- delrel у значков для щ и ж - спорно